# 🚗 Driver Monitoring System — THS2026-77
**Đại học Cần Thơ · Trường CNTT-TT · 2026**

**Pipeline:** Live Video → EAR/MAR/HeadPose/Gaze → Bi-LSTM → Attention Score 0-100%

## Setup: Attach datasets cần thiết
1. `nthu-ddd` — drowsy driving videos
2. `mrl-eye-dataset` — eye open/closed images
3. `driver-monitoring-system` — this project's code (uploaded as Kaggle dataset)

In [ ]:
import os, sys
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU')

# Kaggle paths
NTHU_PATH  = '/kaggle/input/nthu-ddd'
MRL_PATH   = '/kaggle/input/mrl-eye-dataset'
CODE_PATH  = '/kaggle/input/driver-monitoring-system'
OUTPUT_DIR = '/kaggle/working'

print('NTHU-DDD exists:', os.path.exists(NTHU_PATH))
print('MRL exists:', os.path.exists(MRL_PATH))
print('Code exists:', os.path.exists(CODE_PATH))

In [ ]:
# Install dependencies
!pip install mediapipe>=1.0.0 scipy scikit-learn seaborn -q

In [ ]:
# Add project to path
sys.path.insert(0, CODE_PATH)
sys.path.insert(0, '/kaggle/input/driver-monitoring-system/driver_monitoring')

# Download MediaPipe FaceLandmarker model
import urllib.request
os.makedirs(f'{OUTPUT_DIR}/models', exist_ok=True)
model_url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
model_path = f'{OUTPUT_DIR}/models/face_landmarker.task'
if not os.path.exists(model_path):
    print('Downloading face_landmarker.task ...')
    urllib.request.urlretrieve(model_url, model_path)
    print(f'Done: {os.path.getsize(model_path)/1e6:.1f} MB')
else:
    print('face_landmarker.task already exists')

## 1. Phân Tích Dataset — Kiểm Tra Bias Trước Khi Train

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from tqdm.notebook import tqdm

# ── Phân tích NTHU-DDD: số video, độ dài, class distribution ────────────
nthu = Path(NTHU_PATH)
stats = []

for split in ['training_data', 'testing_data']:
    for label in ['drowsy', 'nondrowsy']:
        folder = nthu / split / label
        if not folder.exists():
            # Try alternative names
            for alt in ['non_drowsy', 'alert', 'non-drowsy']:
                alt_folder = nthu / split / alt
                if alt_folder.exists():
                    folder = alt_folder
                    break
        if not folder.exists():
            continue
        videos = list(folder.glob('*.avi')) + list(folder.glob('*.mp4'))
        for vp in videos:
            cap = cv2.VideoCapture(str(vp))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            stats.append({'split': split, 'label': label, 'video': vp.name,
                           'fps': fps, 'frames': frames, 'duration_s': frames/fps})

df_nthu = pd.DataFrame(stats)
print(f'Total videos: {len(df_nthu)}')
print(df_nthu.groupby(['split', 'label'])[['frames', 'duration_s']].describe().round(1))

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Video count
df_nthu.groupby(['split', 'label']).size().unstack().plot(kind='bar', ax=axes[0], color=['#E74C3C','#2ECC71'])
axes[0].set_title('Video Count (Class Imbalance Check)')
axes[0].set_xlabel('Split')

# Duration distribution
df_nthu.boxplot(column='duration_s', by='label', ax=axes[1])
axes[1].set_title('Video Duration (seconds)')

# Total minutes per class
total_mins = df_nthu.groupby('label')['duration_s'].sum() / 60
total_mins.plot(kind='bar', ax=axes[2], color=['#E74C3C','#2ECC71'])
axes[2].set_title('Total Duration per Class (minutes)')

plt.suptitle('NTHU-DDD Dataset Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nthu_analysis.png', dpi=150)
plt.show()
print('Saved: nthu_analysis.png')

## 2. Extract Geometric Features từ NTHU-DDD
**Tại sao không dùng pixel?** → Chống học vẹt background, ánh sáng, khoảng cách camera

In [ ]:
# Patch paths for Kaggle
import importlib, core.face_analyzer as fa_module
fa_module._MODEL_PATH = Path(model_path)

from core.face_analyzer import FaceAnalyzer, FaceFeatures
from training.dataset_builder import DatasetBuilder, FEATURE_NAMES, SEQ_LEN

os.makedirs(f'{OUTPUT_DIR}/data/processed', exist_ok=True)

builder = DatasetBuilder(output_dir=f'{OUTPUT_DIR}/data/processed')

print('Extracting features from NTHU-DDD...')
print('This will take 10-30 minutes depending on dataset size...')

X_train, y_train = builder.process_nthu_ddd(NTHU_PATH, split='train')
X_test,  y_test  = builder.process_nthu_ddd(NTHU_PATH, split='test')

print(f'\nTrain: {len(X_train)} sequences, Test: {len(X_test)} sequences')
print(f'Feature shape: {X_train.shape}')

In [ ]:
# === PHÂN TÍCH BIAS ===
if len(X_train) > 0:
    bias_report = builder.analyze_dataset_bias(X_train, y_train)

    # Visualize feature distributions per class
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    axes = axes.flatten()

    class_names = ['Alert', 'Drowsy']
    colors = ['#2ECC71', '#E74C3C']

    for feat_idx, fname in enumerate(FEATURE_NAMES[:12]):
        ax = axes[feat_idx]
        for cls_id, (cls_name, color) in enumerate(zip(class_names, colors)):
            mask = y_train == cls_id
            if mask.sum() > 0:
                vals = X_train[mask, :, feat_idx].flatten()
                ax.hist(vals, bins=40, alpha=0.6, color=color,
                        label=cls_name, density=True)
        ax.set_title(fname, fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.suptitle('Feature Distributions per Class (Overlap = Harder to learn = Less overfitting risk)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/feature_distributions.png', dpi=150)
    plt.show()
    print('Saved: feature_distributions.png')

In [ ]:
# === AUGMENTATION ===
print('Augmenting training data...')
X_aug, y_aug = builder.augment_sequences(X_train, y_train)
print(f'After augmentation: {len(X_aug)} sequences')
print(f'Class distribution: {np.bincount(y_aug)}')

# Save
builder.save_dataset(X_aug, y_aug, 'train')
builder.save_dataset(X_test, y_test, 'test')
print('Saved to:', f'{OUTPUT_DIR}/data/processed/')

## 3. Train Bi-LSTM Classifier

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from models.lstm_classifier import DriverStateLSTM, ModelManager, CLASSES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Load data
X_tr = np.load(f'{OUTPUT_DIR}/data/processed/X_train.npy')
y_tr = np.load(f'{OUTPUT_DIR}/data/processed/y_train.npy')

# ⚠️ Subject-disjoint split: KHÔNG random split từ cùng video
# Dùng test set đã extract riêng từ testing_data của NTHU-DDD
X_te = np.load(f'{OUTPUT_DIR}/data/processed/X_test.npy') \
    if os.path.exists(f'{OUTPUT_DIR}/data/processed/X_test.npy') else None

# Validation split từ train (stratified)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42)

print(f'Train: {len(X_tr2)} | Val: {len(X_val)} | Test: {len(X_te) if X_te is not None else "N/A"}')

In [ ]:
# DataLoaders với WeightedRandomSampler (chống class imbalance)
X_tr_t  = torch.FloatTensor(X_tr2).to(device)
y_tr_t  = torch.LongTensor(y_tr2).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.LongTensor(y_val).to(device)

counts = np.bincount(y_tr2)
weights_per_class = 1.0 / (counts + 1e-6)
sample_weights = torch.FloatTensor([weights_per_class[y] for y in y_tr2])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_ds = TensorDataset(X_tr_t, y_tr_t)
val_ds   = TensorDataset(X_val_t, y_val_t)
train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler)
val_loader   = DataLoader(val_ds,   batch_size=64)

print(f'Class counts: {dict(zip(CLASSES[:2], counts[:2]))}')

In [ ]:
# Model + Loss + Optimizer
model = DriverStateLSTM().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

# Class-weighted loss: penalize more for missing drowsy
class_weights = torch.FloatTensor(
    [len(y_tr2) / (len(CLASSES) * c + 1e-6) for c in counts]
).to(device)
# Double weight for DROWSY (safety-critical)
if len(class_weights) > 1:
    class_weights[1] *= 2.0

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)

In [ ]:
# Training loop
EPOCHS  = 60
PATIENCE = 12
best_val_acc = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

os.makedirs(f'{OUTPUT_DIR}/models/saved', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        logits, _ = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item() * len(yb)
        t_correct += (logits.argmax(1) == yb).sum().item()
        t_total += len(yb)

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            logits, _ = model(Xb)
            loss = criterion(logits, yb)
            v_loss += loss.item() * len(yb)
            v_correct += (logits.argmax(1) == yb).sum().item()
            v_total += len(yb)

    scheduler.step()
    t_acc = t_correct / t_total * 100
    v_acc = v_correct / v_total * 100
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train: {t_loss/t_total:.4f} ({t_acc:.1f}%) | '
              f'Val: {v_loss/v_total:.4f} ({v_acc:.1f}%)')

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict(),
                    'val_acc': v_acc, 'epoch': epoch,
                    'config': {'feature_dim': 12, 'seq_len': 30, 'classes': CLASSES}},
                   f'{OUTPUT_DIR}/models/saved/driver_lstm_nthu.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\n✅ Best Val Accuracy: {best_val_acc:.1f}%')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train', color='#4A90D9')
ax1.plot(history['val_loss'],   label='Val',   color='#E74C3C')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train', color='#4A90D9')
ax2.plot(history['val_acc'],   label='Val',   color='#E74C3C')
ax2.set_title('Accuracy (%)'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f'Training Curves — Best Val: {best_val_acc:.1f}%', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()

## 4. Đánh Giá Trên Test Set (Subject-Disjoint)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Load best model
checkpoint = torch.load(f'{OUTPUT_DIR}/models/saved/driver_lstm_nthu.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Test on held-out set
if X_te is not None and len(X_te) > 0:
    X_te_t = torch.FloatTensor(X_te).to(device)
    with torch.no_grad():
        logits, _ = model(X_te_t)
        y_pred = logits.argmax(1).cpu().numpy()

    print('=== TEST SET RESULTS (Subject-Disjoint) ===')
    print(classification_report(y_test, y_pred,
                                 target_names=CLASSES[:len(np.unique(y_test))], digits=3))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-6)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES[:cm.shape[0]],
                yticklabels=CLASSES[:cm.shape[0]], ax=ax)
    ax.set_title(f'Confusion Matrix (normalized) — {(y_pred==y_test).mean()*100:.1f}% acc')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150)
    plt.show()
else:
    print('No separate test set available — evaluate on val set')

## 5. Sanity Check — Kiểm Tra Model Có Học Vẹt Không

In [ ]:
# Permutation test: nhãn ngẫu nhiên → accuracy phải về ~50% (binary) hoặc ~25% (4 class)
if X_te is not None and len(X_te) > 0:
    X_sample = X_te[:400]
    y_sample = y_test[:400]
else:
    X_sample = X_val[:400]
    y_sample = y_val[:400]

y_random = np.random.permutation(y_sample)
X_s_t = torch.FloatTensor(X_sample).to(device)
with torch.no_grad():
    preds = model(X_s_t)[0].argmax(1).cpu().numpy()

from sklearn.metrics import accuracy_score
real_acc   = accuracy_score(y_sample, preds)
random_acc = accuracy_score(y_random, preds)
print(f'Real labels accuracy:   {real_acc*100:.1f}%')
print(f'Random labels accuracy: {random_acc*100:.1f}%')
print()
n_classes = len(np.unique(y_sample))
chance = 100 / n_classes
if random_acc * 100 < chance + 5:
    print(f'✅ PASS: Random acc ({random_acc*100:.1f}%) ≈ chance ({chance:.0f}%) — Không có data leakage')
else:
    print(f'⚠️ WARNING: Random acc cao bất thường — kiểm tra data leakage!')

In [ ]:
# Feature importance (ablation)
feature_names = ['EAR_L','EAR_R','EAR_avg','MAR','Yaw','Pitch','Roll',
                  'Gaze_X','Gaze_Y','PERCLOS','EyesClosed','MouthOpen']

with torch.no_grad():
    base_preds = model(X_s_t)[0].argmax(1).cpu().numpy()
base_acc = accuracy_score(y_sample, base_preds)

importances = {}
for fi, fname in enumerate(feature_names):
    X_abl = X_sample.copy()
    X_abl[:, :, fi] = 0.0
    with torch.no_grad():
        abl_preds = model(torch.FloatTensor(X_abl).to(device))[0].argmax(1).cpu().numpy()
    abl_acc = accuracy_score(y_sample, abl_preds)
    importances[fname] = base_acc - abl_acc

imp_df = pd.DataFrame(list(importances.items()), columns=['Feature', 'Importance']).sort_values('Importance', ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#E74C3C' if v > 0.05 else ('#F39C12' if v > 0.01 else '#95A5A6')
           for v in imp_df['Importance']]
ax.barh(imp_df['Feature'], imp_df['Importance'] * 100, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Accuracy drop when feature zeroed out (%)')
ax.set_title('Feature Importance via Ablation')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_importance.png', dpi=150)
plt.show()
print('Red = important | Orange = moderate | Gray = less important')

## 6. Export Model

In [ ]:
import time, json

# Benchmark latency
model.eval()
dummy = torch.zeros(1, 30, 12).to(device)
with torch.no_grad():
    for _ in range(50): model(dummy)  # warmup

N = 500
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N): model(dummy)
latency_ms = (time.perf_counter() - t0) / N * 1000

print(f'Latency: {latency_ms:.2f} ms/frame → {1000/latency_ms:.0f} FPS')
print(f'Model size: {os.path.getsize(f"{OUTPUT_DIR}/models/saved/driver_lstm_nthu.pt")/1e6:.1f} MB')

# Save metadata
meta = {
    'model_name': 'driver_lstm_nthu',
    'best_val_acc': best_val_acc,
    'latency_ms': latency_ms,
    'classes': CLASSES,
    'feature_names': feature_names,
    'seq_len': 30,
    'feature_dim': 12,
    'architecture': 'Bidirectional LSTM + Self-Attention',
    'dataset': 'NTHU-DDD + Augmentation',
    'anti_bias': 'geometric features only, subject-disjoint split, weighted loss'
}
with open(f'{OUTPUT_DIR}/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('\nAll outputs saved to /kaggle/working/')
for f in Path(OUTPUT_DIR).rglob('*'):
    if f.is_file() and '.pt' not in str(f) or 'lstm' in str(f):
        print(f'  {f.relative_to(OUTPUT_DIR)}')